# Data Preprocessing Phase 1 — CLEAN

Notebook nay phu trach Layer 1 cua pipeline.

Dau ra mong muon:
- `data/clean/products.parquet`
- `data/clean/customers.parquet`
- `data/clean/promotions.parquet`
- `data/clean/geography.parquet`
- `data/clean/orders.parquet`
- `data/clean/order_items.parquet`
- `data/clean/payments.parquet`
- `data/clean/shipments.parquet`
- `data/clean/returns.parquet`
- `data/clean/reviews.parquet`
- `data/clean/sales.parquet`
- `data/clean/inventory.parquet`
- `data/clean/web_traffic.parquet`
- `data/clean/sample_submission.parquet`

Notebook chi lam cac viec cua CLEAN layer:
- enforce dtype
- chuan hoa string
- xu ly duplicate exact-match da xac nhan o `order_items`
- xu ly business logic nulls
- giu nguyen nghiep vu `shipments` thieu record
- xuat toan bo bang clean sang `data/clean/*.parquet`

Notebook nay KHONG lam feature engineering.


In [1]:
import warnings
from importlib.util import find_spec
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 200)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')

def find_project_root():
    cwd = Path.cwd().resolve()
    candidate_roots = [cwd, cwd / 'VinDatathon_the-4-Outliers', *cwd.parents]
    try:
        candidate_roots.extend(path for path in cwd.iterdir() if path.is_dir())
    except OSError:
        pass

    seen = set()
    for candidate in candidate_roots:
        candidate = candidate.resolve()
        if candidate in seen:
            continue
        seen.add(candidate)
        if (candidate / 'datathon-2026-round-1').exists():
            return candidate
    raise FileNotFoundError(
        "Khong tim thay thu muc project chua `datathon-2026-round-1/`. "
        "Hay mo notebook tu `VinDatathon_the-4-Outliers` hoac repo root."
    )

def detect_parquet_engine():
    if find_spec('pyarrow') is not None:
        return 'pyarrow'
    if find_spec('fastparquet') is not None:
        return 'fastparquet'
    return None

ROOT = find_project_root()
RAW_DIR = ROOT / 'datathon-2026-round-1'
CLEAN_DIR = ROOT / 'data' / 'clean'
CLEAN_DIR.mkdir(parents=True, exist_ok=True)
PARQUET_ENGINE = detect_parquet_engine()

print(f'[INFO] Project root: {ROOT}')
if PARQUET_ENGINE is None:
    print('[INFO] Khong co pyarrow/fastparquet -> Phase 1 se ghi pickle fallback thay vi crash.')
else:
    print(f'[INFO] Parquet engine: {PARQUET_ENGINE}')

DATE_COLUMNS = {
    'customers': ['signup_date'],
    'promotions': ['start_date', 'end_date'],
    'orders': ['order_date'],
    'shipments': ['ship_date', 'delivery_date'],
    'returns': ['return_date'],
    'reviews': ['review_date'],
    'sales': ['Date'],
    'inventory': ['snapshot_date'],
    'web_traffic': ['date']
}

EXPECTED_TABLES = [
    'products', 'customers', 'promotions', 'geography', 'orders', 'order_items',
    'payments', 'shipments', 'returns', 'reviews', 'sales', 'inventory',
    'web_traffic', 'sample_submission'
]


[INFO] Project root: D:\B. COMPUTER SCIENCE PROJECTS\D. DATATHON\VinDatathon_the-4-Outliers
[INFO] Parquet engine: pyarrow


In [2]:
def read_table(name):
    path = RAW_DIR / f'{name}.csv'
    if not path.exists():
        raise FileNotFoundError(f'Missing raw table: {path}')
    return pd.read_csv(path, parse_dates=DATE_COLUMNS.get(name))

def normalize_text(df, keep_case_cols=None):
    keep_case_cols = set() if keep_case_cols is None else set(keep_case_cols)
    obj_cols = [c for c in df.columns if df[c].dtype == 'object']
    for col in obj_cols:
        df[col] = df[col].astype('string').str.strip().str.replace(r'\s+', ' ', regex=True)
        if col not in keep_case_cols:
            df[col] = df[col]
    return df

def enforce_numeric(df, cols):
    for col in cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    return df

def save_clean_table(df, name):
    parquet_path = CLEAN_DIR / f'{name}.parquet'
    if PARQUET_ENGINE is not None:
        try:
            df.to_parquet(parquet_path, index=False, engine=PARQUET_ENGINE)
            return parquet_path
        except Exception as e:
            print(f'[WARN] Ghi parquet loi voi {name}: {e}')
    fallback_path = CLEAN_DIR / f'{name}.pkl'
    df.to_pickle(fallback_path)
    print(f'[FALLBACK] Da ghi pickle: {fallback_path.name}')
    return fallback_path

summary_rows = []
def add_summary(table, metric, value):
    summary_rows.append({'table': table, 'metric': metric, 'value': value})


## 1. Doc tat ca cac bang raw

Neu team bo sung them file CSV trong raw folder thi can cap nhat danh sach `EXPECTED_TABLES` o tren.


In [3]:
raw = {name: read_table(name) for name in EXPECTED_TABLES}

raw_overview = pd.DataFrame([
    {'table': name, 'rows': df.shape[0], 'cols': df.shape[1]}
    for name, df in raw.items()
]).sort_values('table').reset_index(drop=True)

raw_overview


,table,rows,cols
0,customers,121930,7
1,geography,39948,4
2,inventory,60247,17
3,order_items,714669,7
4,orders,646945,8
5,payments,646945,4
6,products,2412,8
7,promotions,50,10
8,returns,39939,7
9,reviews,113551,7


## 2. Chuan hoa co ban

Layer 1 chi lam sach du lieu, chua tinh metric business.


In [4]:
display_text_cols = {'product_name', 'promo_name', 'city', 'district', 'review_title'}

products_clean = normalize_text(raw['products'].copy(), keep_case_cols=display_text_cols)
customers_clean = normalize_text(raw['customers'].copy(), keep_case_cols=display_text_cols)
promotions_clean = normalize_text(raw['promotions'].copy(), keep_case_cols=display_text_cols)
geography_clean = normalize_text(raw['geography'].copy(), keep_case_cols=display_text_cols)
orders_clean = normalize_text(raw['orders'].copy(), keep_case_cols=display_text_cols)
order_items_clean = normalize_text(raw['order_items'].copy(), keep_case_cols=display_text_cols)
payments_clean = normalize_text(raw['payments'].copy(), keep_case_cols=display_text_cols)
shipments_clean = normalize_text(raw['shipments'].copy(), keep_case_cols=display_text_cols)
returns_clean = normalize_text(raw['returns'].copy(), keep_case_cols=display_text_cols)
reviews_clean = normalize_text(raw['reviews'].copy(), keep_case_cols=display_text_cols)
sales_clean = normalize_text(raw['sales'].copy(), keep_case_cols=display_text_cols)
inventory_clean = normalize_text(raw['inventory'].copy(), keep_case_cols=display_text_cols)
web_traffic_clean = normalize_text(raw['web_traffic'].copy(), keep_case_cols=display_text_cols)
sample_submission_clean = normalize_text(raw['sample_submission'].copy(), keep_case_cols=display_text_cols)

if 'Date' in sales_clean.columns:
    sales_clean = sales_clean.rename(columns={'Date': 'date'})

products_clean = enforce_numeric(products_clean, ['product_id', 'price', 'cogs'])
customers_clean = enforce_numeric(customers_clean, ['customer_id', 'zip'])
promotions_clean = enforce_numeric(promotions_clean, ['discount_value', 'stackable_flag', 'min_order_value'])
geography_clean = enforce_numeric(geography_clean, ['zip'])
orders_clean = enforce_numeric(orders_clean, ['order_id', 'customer_id', 'zip'])
order_items_clean = enforce_numeric(order_items_clean, ['order_id', 'product_id', 'quantity', 'unit_price', 'discount_amount'])
payments_clean = enforce_numeric(payments_clean, ['order_id', 'payment_value', 'installments'])
shipments_clean = enforce_numeric(shipments_clean, ['order_id', 'shipping_fee'])
returns_clean = enforce_numeric(returns_clean, ['order_id', 'product_id', 'return_quantity', 'refund_amount'])
reviews_clean = enforce_numeric(reviews_clean, ['order_id', 'product_id', 'customer_id', 'rating'])
sales_clean = enforce_numeric(sales_clean, ['Revenue', 'COGS'])
inventory_clean = enforce_numeric(inventory_clean, ['product_id', 'stock_on_hand', 'units_received', 'units_sold', 'stockout_days', 'days_of_supply', 'fill_rate', 'stockout_flag', 'overstock_flag', 'reorder_flag', 'sell_through_rate', 'year', 'month'])
web_traffic_clean = enforce_numeric(web_traffic_clean, ['sessions', 'unique_visitors', 'page_views', 'bounce_rate', 'avg_session_duration_sec'])
sample_submission_clean = enforce_numeric(sample_submission_clean, [c for c in sample_submission_clean.columns if c.lower() != 'date'])

add_summary('all', 'basic_cleaning', 'done')


## 3. Duplicate handling

Rule cua Phase 1:
- Chi xu ly duplicate exact-match da xac nhan o `order_items`
- Cac bang khac chi audit, khong drop bua


In [5]:
order_items_exact_dup = order_items_clean[order_items_clean.duplicated(keep='first')].copy()
n_order_items_exact_dup = int(len(order_items_exact_dup))
order_items_clean = order_items_clean.drop_duplicates().reset_index(drop=True)

duplicate_audit = pd.DataFrame([
    {'table': 'products', 'duplicate_rows_on_key': int(products_clean.duplicated(subset=['product_id'], keep=False).sum())},
    {'table': 'customers', 'duplicate_rows_on_key': int(customers_clean.duplicated(subset=['customer_id'], keep=False).sum())},
    {'table': 'promotions', 'duplicate_rows_on_key': int(promotions_clean.duplicated(subset=['promo_id'], keep=False).sum())},
    {'table': 'geography', 'duplicate_rows_on_key': int(geography_clean.duplicated(subset=['zip'], keep=False).sum())},
    {'table': 'orders', 'duplicate_rows_on_key': int(orders_clean.duplicated(subset=['order_id'], keep=False).sum())},
    {'table': 'payments', 'duplicate_rows_on_key': int(payments_clean.duplicated(subset=['order_id'], keep=False).sum())},
    {'table': 'shipments', 'duplicate_rows_on_key': int(shipments_clean.duplicated(subset=['order_id'], keep=False).sum())},
    {'table': 'returns', 'duplicate_rows_on_key': int(returns_clean.duplicated(subset=['return_id'], keep=False).sum())},
    {'table': 'reviews', 'duplicate_rows_on_key': int(reviews_clean.duplicated(subset=['review_id'], keep=False).sum())}
])

add_summary('order_items', 'exact_duplicates_removed', n_order_items_exact_dup)
duplicate_audit


,table,duplicate_rows_on_key
0,products,0
1,customers,0
2,promotions,0
3,geography,0
4,orders,0
5,payments,0
6,shipments,0
7,returns,0
8,reviews,0


## 4. Business logic nulls

Cac missing duoc xu ly theo dung nghiep vu da chot truoc do.


In [6]:
promotions_clean['applicable_category'] = promotions_clean['applicable_category'].fillna('All_Categories')
order_items_clean['promo_id'] = order_items_clean['promo_id'].fillna('No_Promo')
customers_clean['gender'] = customers_clean['gender'].fillna('Not_Provided')
customers_clean['age_group'] = customers_clean['age_group'].fillna('Not_Provided')
customers_clean['acquisition_channel'] = customers_clean['acquisition_channel'].fillna('Unknown_Source')

add_summary('promotions', 'applicable_category_fillna', 'All_Categories')
add_summary('order_items', 'promo_id_fillna', 'No_Promo')
add_summary('customers', 'gender_fillna', 'Not_Provided')
add_summary('customers', 'age_group_fillna', 'Not_Provided')
add_summary('customers', 'acquisition_channel_fillna', 'Unknown_Source')


## 5. Shipments missing records la business logic

Khong fill `shipping_fee`, khong tao shipment gia.
Chi tao bang audit de team nhin ro dau la missing hop ly theo nghiep vu.


In [7]:
orders_shipments_audit = orders_clean.merge(
    shipments_clean[['order_id', 'ship_date', 'delivery_date', 'shipping_fee']],
    on='order_id',
    how='left'
)
orders_shipments_audit['has_shipment_record'] = orders_shipments_audit['ship_date'].notna().astype('int8')
orders_shipments_audit['shipment_expected_flag'] = orders_shipments_audit['order_status'].isin(['shipped', 'delivered', 'returned']).astype('int8')
orders_shipments_audit['shipment_missing_is_business_logic_flag'] = (
    (orders_shipments_audit['has_shipment_record'] == 0)
    & (orders_shipments_audit['shipment_expected_flag'] == 0)
).astype('int8')
orders_shipments_audit['shipment_missing_unexpected_flag'] = (
    (orders_shipments_audit['has_shipment_record'] == 0)
    & (orders_shipments_audit['shipment_expected_flag'] == 1)
).astype('int8')

add_summary('shipments', 'missing_records_vs_orders', int(len(orders_clean) - len(shipments_clean)))
add_summary('shipments', 'missing_records_business_logic', int(orders_shipments_audit['shipment_missing_is_business_logic_flag'].sum()))
add_summary('shipments', 'missing_records_unexpected', int(orders_shipments_audit['shipment_missing_unexpected_flag'].sum()))

orders_shipments_audit[['order_status', 'shipment_missing_is_business_logic_flag']].groupby('order_status', as_index=False).sum().sort_values('shipment_missing_is_business_logic_flag', ascending=False)


,order_status,shipment_missing_is_business_logic_flag
0,cancelled,59462
3,paid,13577
1,created,7275
2,delivered,0
4,returned,0
5,shipped,0


## 6. Logic checks va audit phu

Phan nay khong tao feature, chi kiem tra tinh hop le co ban de team de debug.


In [8]:
logic_audit = pd.DataFrame([
    {'table': 'products', 'metric': 'invalid_cogs_ge_price', 'value': int((products_clean['cogs'] >= products_clean['price']).sum())},
    {'table': 'reviews', 'metric': 'invalid_rating_out_of_range', 'value': int((~reviews_clean['rating'].between(1, 5)).sum())},
    {'table': 'web_traffic', 'metric': 'invalid_bounce_rate_out_of_range', 'value': int((~web_traffic_clean['bounce_rate'].between(0, 1)).sum())}
])

all_product_ids = set(products_clean['product_id'])
active_product_ids = set(order_items_clean['product_id'].dropna().astype(int))
ghost_product_ids = all_product_ids - active_product_ids
active_min_price = products_clean.loc[products_clean['product_id'].isin(active_product_ids), 'price'].min()
ghost_fake_threshold = 0.75 * active_min_price

ghost_products_audit = products_clean.loc[products_clean['product_id'].isin(ghost_product_ids)].copy()
ghost_products_audit['ghost_type'] = np.where(ghost_products_audit['price'] < ghost_fake_threshold, 'ghost_fake', 'ghost_real')

add_summary('products', 'ghost_products_total', int(len(ghost_products_audit)))
add_summary('products', 'ghost_fake', int((ghost_products_audit['ghost_type'] == 'ghost_fake').sum()))
add_summary('products', 'ghost_real', int((ghost_products_audit['ghost_type'] == 'ghost_real').sum()))

logic_audit


,table,metric,value
0,products,invalid_cogs_ge_price,0
1,reviews,invalid_rating_out_of_range,0
2,web_traffic,invalid_bounce_rate_out_of_range,0


## 7. Export to `data/clean/*.parquet`

Day la output chinh cua Phase 1.
Neu may bao loi khi ghi parquet, can cai them `pyarrow` hoac `fastparquet` trong kernel notebook.


In [9]:
export_logs = [
    {'dataset': 'products', 'path': str(save_clean_table(products_clean, 'products'))},
    {'dataset': 'customers', 'path': str(save_clean_table(customers_clean, 'customers'))},
    {'dataset': 'promotions', 'path': str(save_clean_table(promotions_clean, 'promotions'))},
    {'dataset': 'geography', 'path': str(save_clean_table(geography_clean, 'geography'))},
    {'dataset': 'orders', 'path': str(save_clean_table(orders_clean, 'orders'))},
    {'dataset': 'order_items', 'path': str(save_clean_table(order_items_clean, 'order_items'))},
    {'dataset': 'payments', 'path': str(save_clean_table(payments_clean, 'payments'))},
    {'dataset': 'shipments', 'path': str(save_clean_table(shipments_clean, 'shipments'))},
    {'dataset': 'returns', 'path': str(save_clean_table(returns_clean, 'returns'))},
    {'dataset': 'reviews', 'path': str(save_clean_table(reviews_clean, 'reviews'))},
    {'dataset': 'sales', 'path': str(save_clean_table(sales_clean, 'sales'))},
    {'dataset': 'inventory', 'path': str(save_clean_table(inventory_clean, 'inventory'))},
    {'dataset': 'web_traffic', 'path': str(save_clean_table(web_traffic_clean, 'web_traffic'))},
    {'dataset': 'sample_submission', 'path': str(save_clean_table(sample_submission_clean, 'sample_submission'))}
]

pd.DataFrame(export_logs)


,dataset,path
0,products,D:\B. COMPUTER SCIENCE PROJECTS\D. DATATHON\Vi...
1,customers,D:\B. COMPUTER SCIENCE PROJECTS\D. DATATHON\Vi...
2,promotions,D:\B. COMPUTER SCIENCE PROJECTS\D. DATATHON\Vi...
3,geography,D:\B. COMPUTER SCIENCE PROJECTS\D. DATATHON\Vi...
4,orders,D:\B. COMPUTER SCIENCE PROJECTS\D. DATATHON\Vi...
5,order_items,D:\B. COMPUTER SCIENCE PROJECTS\D. DATATHON\Vi...
6,payments,D:\B. COMPUTER SCIENCE PROJECTS\D. DATATHON\Vi...
7,shipments,D:\B. COMPUTER SCIENCE PROJECTS\D. DATATHON\Vi...
8,returns,D:\B. COMPUTER SCIENCE PROJECTS\D. DATATHON\Vi...
9,reviews,D:\B. COMPUTER SCIENCE PROJECTS\D. DATATHON\Vi...


In [10]:
clean_summary = pd.DataFrame(summary_rows)
clean_summary


,table,metric,value
0,all,basic_cleaning,done
1,order_items,exact_duplicates_removed,0
2,promotions,applicable_category_fillna,All_Categories
3,order_items,promo_id_fillna,No_Promo
4,customers,gender_fillna,Not_Provided
5,customers,age_group_fillna,Not_Provided
6,customers,acquisition_channel_fillna,Unknown_Source
7,shipments,missing_records_vs_orders,80878
8,shipments,missing_records_business_logic,80314
9,shipments,missing_records_unexpected,564


## 8. Phase 1 handoff - insights va downstream caveats

Section nay chot lai nhung insight cua CLEAN layer duoi dang hop dong du lieu cho cac phase sau.
Muc tieu la de EDA, feature engineering, modelling va report deu dung cung mot cach hieu ve du lieu da clean.


In [11]:
order_items_key_dup = order_items_clean[
    order_items_clean.duplicated(subset=['order_id', 'product_id'], keep=False)
].copy()
order_items_key_dup_groups = order_items_key_dup[['order_id', 'product_id']].drop_duplicates().shape[0]

promo1_missing = int(raw['order_items']['promo_id'].isna().sum())
promo2_non_missing = int(raw['order_items']['promo_id_2'].notna().sum())
promo2_missing = int(raw['order_items']['promo_id_2'].isna().sum())
applicable_category_missing = int(raw['promotions']['applicable_category'].isna().sum())

shipment_gap = int(len(orders_clean) - len(shipments_clean))
shipment_gap_biz = int(orders_shipments_audit['shipment_missing_is_business_logic_flag'].sum())
shipment_gap_unexpected = int(orders_shipments_audit['shipment_missing_unexpected_flag'].sum())
shipment_unexpected_by_status = (
    orders_shipments_audit.loc[
        orders_shipments_audit['shipment_missing_unexpected_flag'] == 1,
        'order_status'
    ]
    .value_counts()
    .to_dict()
)

sales_min_date = sales_clean['date'].min().date()
sales_max_date = sales_clean['date'].max().date()
web_min_date = web_traffic_clean['date'].min().date()
web_max_date = web_traffic_clean['date'].max().date()
web_early_gap_days = int((sales_clean['date'] < pd.Timestamp(web_min_date)).sum())

phase1_key_facts = pd.DataFrame([
    {
        'topic': 'clean_layer_scope',
        'evidence': f"{len(raw)} tables -> {len(export_logs)} parquet outputs",
        'why_it_matters': 'Downstream notebooks nen doc data/clean thay vi quay lai raw CSV.'
    },
    {
        'topic': 'schema_contract',
        'evidence': 'sales.Date da duoc doi ten thanh sales.date; date columns da parse datetime',
        'why_it_matters': 'Join/merge theo ngay nen dung cot date trong clean layer de tranh lech schema.'
    },
    {
        'topic': 'duplicate_contract',
        'evidence': f"order_items exact duplicate removed = {n_order_items_exact_dup}; same (order_id, product_id) groups kept = {order_items_key_dup_groups}",
        'why_it_matters': 'Khong duoc dedup theo (order_id, product_id) mot cach may moc vi van co nhieu line hop le.'
    },
    {
        'topic': 'promo_coverage',
        'evidence': f"promo_id missing in raw = {promo1_missing:,}/{len(raw['order_items']):,} ({promo1_missing / len(raw['order_items']) * 100:.2f}%); promo_id_2 non-null = {promo2_non_missing:,}",
        'why_it_matters': "Sau Phase 1, promo_id null da thanh 'No_Promo'; promo_id_2 van de null de bao toan stacked-promo signal."
    },
    {
        'topic': 'promotion_scope',
        'evidence': f"promotions.applicable_category missing in raw = {applicable_category_missing}/{len(raw['promotions'])} -> fill 'All_Categories'",
        'why_it_matters': 'All_Categories la business sentinel, khong phai missing value can bo qua khi nhom promo.'
    },
    {
        'topic': 'shipment_gap',
        'evidence': f"missing shipment rows = {shipment_gap:,}; business logic = {shipment_gap_biz:,} ({shipment_gap_biz / shipment_gap * 100:.2f}%); unexpected = {shipment_gap_unexpected:,} ({shipment_gap_unexpected / shipment_gap * 100:.2f}%)",
        'why_it_matters': 'NaN o shipments chu yeu la nghiep vu; inner join orders x shipments se lam rot mot so don da fulfilled.'
    },
    {
        'topic': 'shipment_unexpected_breakdown',
        'evidence': f"unexpected missing shipments by status = {shipment_unexpected_by_status}",
        'why_it_matters': 'Can audit rieng delivered/returned/shipped khong co shipment truoc khi ket luan ve logistics.'
    },
    {
        'topic': 'ghost_products',
        'evidence': f"ghost products = {len(ghost_products_audit):,}/{len(products_clean):,} ({len(ghost_products_audit) / len(products_clean) * 100:.2f}%); ghost_fake = {int((ghost_products_audit['ghost_type'] == 'ghost_fake').sum()):,}; ghost_real = {int((ghost_products_audit['ghost_type'] == 'ghost_real').sum()):,}",
        'why_it_matters': 'Ghost products khong vao order_items; can tach ro inventory/product coverage audit voi transaction analysis.'
    },
    {
        'topic': 'domain_checks',
        'evidence': f"invalid cogs>=price = {int((products_clean['cogs'] >= products_clean['price']).sum())}; invalid rating = {int((~reviews_clean['rating'].between(1, 5)).sum())}; invalid bounce_rate = {int((~web_traffic_clean['bounce_rate'].between(0, 1)).sum())}",
        'why_it_matters': 'Neu cac vi pham nay xuat hien o phase sau, loi nam o join/feature engineering chu khong phai clean layer.'
    },
    {
        'topic': 'temporal_coverage',
        'evidence': f"sales daily range = {sales_min_date} -> {sales_max_date} ({len(sales_clean)} ngay); web_traffic range = {web_min_date} -> {web_max_date} ({len(web_traffic_clean)} ngay); gap truoc web = {web_early_gap_days} ngay",
        'why_it_matters': 'Feature theo traffic can co chien luoc xu ly phan dau 2012 khi web_traffic chua co du lieu.'
    },
    {
        'topic': 'text_normalization_contract',
        'evidence': 'Phase 1 trim whitespace va chuan hoa spacing, nhung khong lowercase text labels',
        'why_it_matters': 'Neu can fuzzy-group text o phase sau, hay tao cot derived rieng thay vi sua cot goc.'
    }
])

phase1_fill_audit = pd.DataFrame([
    {
        'field': 'promotions.applicable_category',
        'raw_missing': applicable_category_missing,
        'share_pct': round(applicable_category_missing / len(raw['promotions']) * 100, 2),
        'clean_value': 'All_Categories',
        'triggered_in_current_snapshot': 'yes'
    },
    {
        'field': 'order_items.promo_id',
        'raw_missing': promo1_missing,
        'share_pct': round(promo1_missing / len(raw['order_items']) * 100, 2),
        'clean_value': 'No_Promo',
        'triggered_in_current_snapshot': 'yes'
    },
    {
        'field': 'order_items.promo_id_2',
        'raw_missing': promo2_missing,
        'share_pct': round(promo2_missing / len(raw['order_items']) * 100, 2),
        'clean_value': 'kept_null',
        'triggered_in_current_snapshot': 'yes'
    },
    {
        'field': 'customers.gender',
        'raw_missing': int(raw['customers']['gender'].isna().sum()),
        'share_pct': round(raw['customers']['gender'].isna().mean() * 100, 2),
        'clean_value': 'Not_Provided',
        'triggered_in_current_snapshot': 'yes' if raw['customers']['gender'].isna().any() else 'no'
    },
    {
        'field': 'customers.age_group',
        'raw_missing': int(raw['customers']['age_group'].isna().sum()),
        'share_pct': round(raw['customers']['age_group'].isna().mean() * 100, 2),
        'clean_value': 'Not_Provided',
        'triggered_in_current_snapshot': 'yes' if raw['customers']['age_group'].isna().any() else 'no'
    },
    {
        'field': 'customers.acquisition_channel',
        'raw_missing': int(raw['customers']['acquisition_channel'].isna().sum()),
        'share_pct': round(raw['customers']['acquisition_channel'].isna().mean() * 100, 2),
        'clean_value': 'Unknown_Source',
        'triggered_in_current_snapshot': 'yes' if raw['customers']['acquisition_channel'].isna().any() else 'no'
    }
])

phase1_downstream_rules = pd.DataFrame([
    {
        'topic': 'read source',
        'do_this': 'Doc tu data/clean/*.parquet lam source-of-truth cho Phase 2 tro di.',
        'avoid_this': 'Doc lai raw CSV roi tu clean lai mot cach khac.',
        'reason': 'De giu on dinh dtype, cot date va sentinel categories.'
    },
    {
        'topic': 'promo flag after clean',
        'do_this': "Dung (promo_id != 'No_Promo') hoac (discount_amount > 0).",
        'avoid_this': 'Dung promo_id.notna() sau Phase 1.',
        'reason': "Vi promo_id da duoc fill thanh 'No_Promo'."
    },
    {
        'topic': 'second promo logic',
        'do_this': 'Dung promo_id_2.notna() de nhan stacked promo, vi cot nay van giu null goc.',
        'avoid_this': 'Fill promo_id_2 hoac gop no vao promo_id chinh.',
        'reason': 'Stacked promo rat hiem va can duoc giu signal rieng.'
    },
    {
        'topic': 'shipment missingness',
        'do_this': 'Xem shipment missing la mot operational state; giu NaN cho shipping columns.',
        'avoid_this': 'Fill shipping_fee/ship_date/delivery_date bang 0, mean, median hoac mode.',
        'reason': 'Phan lon missing la business logic, khong phai data error.'
    },
    {
        'topic': 'joins with shipments',
        'do_this': 'LEFT JOIN tu orders/order_items sang shipments va audit phan unexpected missing.',
        'avoid_this': 'INNER JOIN neu dang tinh funnel, delay, refund, margin hay ops KPI.',
        'reason': 'Van co delivered/returned/shipped khong co shipment record.'
    },
    {
        'topic': 'order_items duplicates',
        'do_this': 'Giu nguyen line-level rows va chi audit duplicate theo business meaning.',
        'avoid_this': 'Drop tat ca rows trung (order_id, product_id).',
        'reason': 'Cung mot san pham trong mot don van co the xuat hien nhieu line hop le voi gia/so luong khac nhau.'
    },
    {
        'topic': 'ghost products',
        'do_this': 'Tach inventory/product coverage audit khoi revenue-order analysis.',
        'avoid_this': 'Mac dinh coi moi product trong products la active SKU da tung ban.',
        'reason': 'Co 814 ghost products, trong do da so la ghost_fake.'
    },
    {
        'topic': 'sentinel categories',
        'do_this': "Giu 'All_Categories', 'No_Promo', 'Not_Provided', 'Unknown_Source' nhu category hop le neu chung xuat hien.",
        'avoid_this': 'Tu dong coi chung la missing va loai bo khoi chart/model.',
        'reason': 'Do la business-safe labels de bao toan traceability cua clean layer.'
    },
    {
        'topic': 'quality regressions',
        'do_this': 'Dung logic_audit hien tai lam baseline khi review phase sau.',
        'avoid_this': 'Giai thich loi domain moi phat sinh nhu mot dac diem cua raw data.',
        'reason': 'Phase 1 da xac nhan 0 vi pham cho cogs/rating/bounce_rate checks.'
    },
    {
        'topic': 'traffic coverage',
        'do_this': 'Ghi ro trong feature notebook cach xu ly 181 ngay dau 2012 khong co web_traffic.',
        'avoid_this': 'Am tham inner join sales voi web_traffic roi de mat ngay train.',
        'reason': 'Traffic chi bat dau tu 2013-01-01, muon forecast clean thi phai xu ly minh bach.'
    }
])

display(phase1_key_facts)
display(phase1_fill_audit)
phase1_downstream_rules


,topic,evidence,why_it_matters
0,clean_layer_scope,14 tables -> 14 parquet outputs,Downstream notebooks nen doc data/clean thay v...
1,schema_contract,sales.Date da duoc doi ten thanh sales.date; d...,Join/merge theo ngay nen dung cot date trong c...
2,duplicate_contract,order_items exact duplicate removed = 0; same ...,"Khong duoc dedup theo (order_id, product_id) m..."
3,promo_coverage,"promo_id missing in raw = 438,353/714,669 (61....","Sau Phase 1, promo_id null da thanh 'No_Promo'..."
4,promotion_scope,promotions.applicable_category missing in raw ...,"All_Categories la business sentinel, khong pha..."
5,shipment_gap,"missing shipment rows = 80,878; business logic...",NaN o shipments chu yeu la nghiep vu; inner jo...
6,shipment_unexpected_breakdown,unexpected missing shipments by status = {'del...,Can audit rieng delivered/returned/shipped kho...
7,ghost_products,"ghost products = 814/2,412 (33.75%); ghost_fak...",Ghost products khong vao order_items; can tach...
8,domain_checks,invalid cogs>=price = 0; invalid rating = 0; i...,"Neu cac vi pham nay xuat hien o phase sau, loi..."
9,temporal_coverage,sales daily range = 2012-07-04 -> 2022-12-31 (...,Feature theo traffic can co chien luoc xu ly p...


,field,raw_missing,share_pct,clean_value,triggered_in_current_snapshot
0,promotions.applicable_category,40,80.0000,All_Categories,yes
1,order_items.promo_id,438353,61.3400,No_Promo,yes
2,order_items.promo_id_2,714463,99.9700,kept_null,yes
3,customers.gender,0,0.0000,Not_Provided,no
4,customers.age_group,0,0.0000,Not_Provided,no
5,customers.acquisition_channel,0,0.0000,Unknown_Source,no


,topic,do_this,avoid_this,reason
0,read source,Doc tu data/clean/*.parquet lam source-of-trut...,Doc lai raw CSV roi tu clean lai mot cach khac.,"De giu on dinh dtype, cot date va sentinel cat..."
1,promo flag after clean,Dung (promo_id != 'No_Promo') hoac (discount_a...,Dung promo_id.notna() sau Phase 1.,Vi promo_id da duoc fill thanh 'No_Promo'.
2,second promo logic,"Dung promo_id_2.notna() de nhan stacked promo,...",Fill promo_id_2 hoac gop no vao promo_id chinh.,Stacked promo rat hiem va can duoc giu signal ...
3,shipment missingness,Xem shipment missing la mot operational state;...,Fill shipping_fee/ship_date/delivery_date bang...,"Phan lon missing la business logic, khong phai..."
4,joins with shipments,LEFT JOIN tu orders/order_items sang shipments...,"INNER JOIN neu dang tinh funnel, delay, refund...",Van co delivered/returned/shipped khong co shi...
5,order_items duplicates,Giu nguyen line-level rows va chi audit duplic...,"Drop tat ca rows trung (order_id, product_id).",Cung mot san pham trong mot don van co the xua...
6,ghost products,Tach inventory/product coverage audit khoi rev...,Mac dinh coi moi product trong products la act...,"Co 814 ghost products, trong do da so la ghost..."
7,sentinel categories,"Giu 'All_Categories', 'No_Promo', 'Not_Provide...",Tu dong coi chung la missing va loai bo khoi c...,Do la business-safe labels de bao toan traceab...
8,quality regressions,Dung logic_audit hien tai lam baseline khi rev...,Giai thich loi domain moi phat sinh nhu mot da...,Phase 1 da xac nhan 0 vi pham cho cogs/rating/...
9,traffic coverage,Ghi ro trong feature notebook cach xu ly 181 n...,Am tham inner join sales voi web_traffic roi d...,"Traffic chi bat dau tu 2013-01-01, muon foreca..."
